<div style="background: linear-gradient(135deg, #12304A, #1F6E8C); color: white; padding: 22px; border-radius: 12px;">
<h2 style="margin-top:0; color:white;">Étude immobilière à Rabat — Analyse exploratoire, nettoyage et prédiction des prix</h2>
<p>Ce notebook présente une démarche complète d’analyse des annonces immobilières de Rabat : exploration des données, nettoyage, préparation des variables, entraînement d’un modèle de prédiction et évaluation de ses performances.</p>
<p><b>Objectif :</b> construire un modèle capable d’estimer le prix d’un bien à partir de ses caractéristiques, tout en documentant les choix méthodologiques et leurs effets sur les données.</p>
</div>

## Organisation du notebook
1. **Compréhension des données** : chargement, diagnostic initial et exploration.
2. **Nettoyage et feature engineering** : harmonisation, valeurs manquantes, doublons et valeurs atypiques.
3. **Prétraitement** : définition de `X` et `y`, regroupement des catégories et encodage.
4. **Modélisation et évaluation** : Random Forest, R², MAE et validation croisée.
5. **Modèle final** : réentraînement, sauvegarde et exemple de prédiction.

Chaque grande partie se termine par une conclusion synthétique. Les graphiques conservent leur interprétation immédiatement en dessous.

**Variables finales du jeu nettoyé :** `Type`, `Localisation`, `Area`, `Rooms`, `Bedrooms`, `Bathrooms`, `Current_state`, `Age`, `city`, `Price`.

> **Convention de lecture :** les montants sont exprimés en dirhams marocains (MAD/DH). Les interprétations sous les graphiques décrivent ce qu’il faut observer sans présumer des résultats chiffrés avant l’exécution du notebook.

<div style="background:#12304A;color:white;padding:16px 20px;border-radius:10px;margin:10px 0;">
<h1 style="margin:0;color:white;">PARTIE I — COMPRÉHENSION DES DONNÉES</h1>
<p style="margin:6px 0 0;">Objectif : découvrir le jeu de données, comprendre ses variables et diagnostiquer sa qualité avant toute transformation.</p>
</div>

## 1. Environnement de travail et paramètres graphiques

Cette cellule importe les bibliothèques utilisées pour la manipulation des données, les visualisations et la modélisation. Elle définit également les paramètres graphiques afin d’assurer une présentation homogène dans tout le rapport.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance


## 2. Chargement des données brutes

Le fichier source est chargé dans un DataFrame. Une copie est conservée avant le nettoyage afin de comparer les données initiales aux données finales et de suivre le nombre d’annonces à chaque étape.

In [ ]:
df = pd.read_csv('mubawab_properties (2).csv')
df_raw = df.copy()            # copie brute pour comparer avant / après
funnel['Données brutes'] = len(df)
print("Shape initiale:", df.shape)
df.head()

### Bilan de l’étape 2

Le chargement fournit le point de départ de l’analyse. La copie `df_raw` permet de conserver une référence intacte pour les comparaisons avant/après.

### Figure 1 — Diagnostic initial de la qualité des données

Cette figure permet d’identifier les colonnes les plus touchées par les valeurs manquantes et de distinguer les variables utiles à l’analyse des champs peu exploitables.

In [ ]:
kept_cols = ['type','area','rooms','bedrooms','bathrooms','property_state','quartier','price']

miss = (df_raw.isna().mean() * 100).sort_values()
miss_df = miss.rename_axis('Colonne').reset_index(name='Pourcentage manquant')
miss_df['Statut'] = miss_df['Colonne'].apply(lambda c: 'Colonne conservée' if c in kept_cols else 'Colonne ignorée')
fig = px.bar(miss_df, x='Pourcentage manquant', y='Colonne', orientation='h', color='Statut',
             title='Valeurs manquantes par colonne (%)',
             color_discrete_map={'Colonne conservée': GREEN, 'Colonne ignorée': GREY})
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_white')
fig.show()

p = pd.to_numeric(df_raw['price'], errors='coerce').dropna()
p = p[p > 0]
fig = px.histogram(x=np.log10(p), nbins=40, title=f'Prix brut (échelle log10) — max = {p.max():.1e} MAD',
                   labels={'x':'log10(Price)', 'y':"Nombre d'annonces"})
fig.update_traces(marker_color=RED, opacity=.85)
fig.update_layout(template='plotly_white')
fig.show()

### Interprétation de la figure 1

Comparer les taux de valeurs manquantes entre les colonnes et vérifier si les variables nécessaires à l’analyse sont suffisamment renseignées. Les champs non retenus peuvent présenter davantage d’absences sans compromettre le modèle, puisqu’ils sont exclus en amont. Le diagnostic doit être relu après exécution avec les pourcentages effectivement affichés.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

## 3. Sélection des variables pertinentes

Les colonnes retenues correspondent aux caractéristiques susceptibles d’expliquer le prix : type de bien, surface, nombre de pièces, état, localisation et prix. Les champs textuels ou techniques non exploités dans le modèle sont écartés.

In [ ]:
cols_needed = ['type','area','rooms','bedrooms','bathrooms','property_state','quartier','price']
df = df[cols_needed].copy()
df.isna().sum()

### Figure 2 — Valeurs manquantes après sélection des variables

La visualisation localise les valeurs absentes dans les colonnes conservées et aide à décider quelles variables nécessitent un traitement avant la modélisation.

In [ ]:
# Carte interactive des valeurs manquantes : 1 = valeur manquante, 0 = présente
missing_map = df.isna().astype(int).reset_index(drop=True)
missing_map.columns = missing_map.columns.astype(str)
fig = px.imshow(missing_map.T, aspect='auto', color_continuous_scale=['#ecf0f1', RED],
                title='Carte des valeurs manquantes (rouge = manquant)',
                labels={'x':'Ligne','y':'Colonne','color':'Manquant'})
fig.update_layout(template='plotly_white', coloraxis_showscale=False)
fig.show()

miss = df.isna().sum().sort_values()
miss_df = miss.rename_axis('Colonne').reset_index(name='Nombre manquant')
miss_df['Pourcentage'] = miss_df['Nombre manquant'] / len(df) * 100
fig = px.bar(miss_df, x='Nombre manquant', y='Colonne', orientation='h', text='Nombre manquant',
             hover_data=['Pourcentage'], title='Nombre de valeurs manquantes')
fig.update_traces(marker_color=RED, textposition='outside')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, xaxis_range=[0, max(1, miss.max()*1.3)], template='plotly_white')
fig.show()

### Interprétation de la figure 2

La carte permet de repérer les colonnes et les lignes comportant des valeurs absentes. Les variables les plus incomplètes demandent une attention particulière : leur traitement dépend de leur rôle dans le modèle et de leur caractère obligatoire ou optionnel. La décision de suppression ou d’imputation est appliquée dans les étapes suivantes.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

<div style="background:#176B67;color:white;padding:16px 20px;border-radius:10px;margin:10px 0;">
<h1 style="margin:0;color:white;">PARTIE II — NETTOYAGE ET FEATURE ENGINEERING</h1>
<p style="margin:6px 0 0;">Objectif : harmoniser les informations, créer les variables utiles, traiter les valeurs manquantes et contrôler les observations atypiques.</p>
</div>

### Conclusion de la partie I — Compréhension des données

Cette première partie permet de prendre connaissance de la structure du fichier, des variables disponibles et de la qualité initiale des données. Les diagnostics visuels servent à repérer les valeurs manquantes, les distributions asymétriques et les informations à conserver pour la suite. **À retenir :** les décisions de nettoyage doivent être guidées par ces constats et documentées.

## 4. Extraction du quartier et de la ville

La colonne de localisation est normalisée puis séparée en deux informations distinctes : le quartier (`Localisation`) et la ville (`city`). Cette étape rend les informations géographiques plus faciles à analyser et à encoder.

In [ ]:
def split_quartier(q):
    if pd.isna(q):
        return (np.nan, np.nan)
    q_clean = ' '.join(str(q).split())        # supprime \n \t multiples
    if ' à ' in q_clean:
        loc, city = q_clean.split(' à ', 1)
        return (loc.strip(), city.strip())
    return (q_clean.strip(), q_clean.strip())

df[['Localisation', 'city']] = df['quartier'].apply(lambda x: pd.Series(split_quartier(x)))
df.drop(columns=['quartier'], inplace=True)
df[['Localisation', 'city']].head()

### Bilan de l’étape 4

La séparation en quartier et ville structure l’information géographique. La cohérence des valeurs extraites doit être vérifiée à l’aide de la visualisation suivante.

### Figure 3 — Répartition géographique des annonces

Les graphiques présentent les localisations les plus fréquentes et la répartition des villes extraites. Ils servent à vérifier que la séparation quartier/ville a produit des catégories cohérentes.

In [ ]:
vc = df['Localisation'].value_counts().head(15).sort_values()
quartiers = vc.rename_axis('Quartier').reset_index(name='Nombre d’annonces')
fig = px.bar(quartiers, x='Nombre d’annonces', y='Quartier', orientation='h',
             title=f"Top 15 quartiers ({df['Localisation'].nunique()} quartiers au total)",
             text='Nombre d’annonces')
fig.update_traces(marker_color=BLUE, textposition='outside')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_white')
fig.show()

vc = df['city'].fillna('Manquant').value_counts().rename_axis('Ville').reset_index(name='Nombre d’annonces')
fig = px.bar(vc, x='Ville', y='Nombre d’annonces', text='Nombre d’annonces', title='Répartition des annonces par ville')
fig.update_traces(marker_color=GREEN, textposition='outside')
fig.update_layout(xaxis_tickangle=-30, template='plotly_white')
fig.show()

### Interprétation de la figure 3

Le graphique des quartiers met en évidence les localisations les plus représentées dans l’échantillon. Le graphique des villes sert à contrôler l’extraction de la ville depuis le champ d’origine. Si une catégorie inattendue ou une valeur manquante apparaît, il convient de vérifier le format source avant de poursuivre.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

## 5. Conversion de l’état du bien en âge estimé

Les tranches d’âge textuelles sont converties en une valeur numérique représentative, généralement le milieu de la tranche. La catégorie d’origine est conservée dans `Current_state` afin de préserver l’information qualitative.

In [ ]:
age_map = {
    "Moins d'un an": 0.5,
    "1-5 ans": 3,
    "5-10 ans": 7.5,
    "10-20 ans": 15,
    "20-30 ans": 25,
    "30-50 ans": 40,
    "50-70 ans": 60,
    "70-100 ans": 85,
    "Plus de 100 ans": 110,
}
df['Age'] = df['property_state'].map(age_map)
df.rename(columns={'property_state': 'Current_state'}, inplace=True)
df[['Current_state', 'Age']].head()

### Bilan de l’étape 5

La variable `Age` rend l’ancienneté exploitable numériquement, tout en gardant la catégorie initiale pour ne pas perdre l’information de départ.

### Figure 4 — État déclaré et âge numérique estimé

Cette figure compare la fréquence des catégories d’état du bien et les valeurs d’âge qui leur sont associées. L’âge obtenu reste une estimation, et non une mesure exacte pour chaque annonce.

In [ ]:
st = df['Current_state'].fillna('Manquant').value_counts()
order = [k for k in age_map if k in st.index] + [k for k in st.index if k not in age_map]
state_df = pd.DataFrame({'Current_state': order, 'Nombre': [st[k] for k in order]})
state_df['Age affecté'] = state_df['Current_state'].apply(lambda o: f"{age_map[o]} ans" if o in age_map else 'Age non défini')
fig = px.bar(state_df, x='Current_state', y='Nombre', color='Age affecté', text='Nombre',
             title='Current_state → Age (valeur affectée)')
fig.update_layout(xaxis_tickangle=-35, template='plotly_white')
fig.show()

fig = px.histogram(df['Age'].dropna(), x='Age', nbins=20,
                   title=f"Distribution de l'âge estimé — {df['Age'].isna().sum()} valeurs manquantes",
                   labels={'Age':'Age (années)', 'count':'Nombre'})
fig.update_traces(marker_color=BLUE)
fig.update_layout(template='plotly_white')
fig.show()

### Interprétation de la figure 4

La distribution des états montre quelles catégories dominent dans les annonces. La conversion en âge numérique facilite l’utilisation de cette information par le modèle, mais elle simplifie nécessairement chaque tranche en une valeur représentative. Il faut donc interpréter `Age` comme une approximation et conserver `Current_state` comme information complémentaire.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

## 6. Harmonisation des noms et de l’ordre des colonnes

Les noms des variables sont harmonisés pour rendre le jeu de données plus lisible et cohérent avec les étapes suivantes. L’ordre des colonnes est fixé pour faciliter l’inspection et l’export.

In [ ]:
df.rename(columns={
    'type': 'Type', 'area': 'Area', 'rooms': 'Rooms',
    'bedrooms': 'Bedrooms', 'bathrooms': 'Bathrooms', 'price': 'Price'
}, inplace=True)

df = df[['Type', 'Localisation', 'Area', 'Rooms', 'Bedrooms', 'Bathrooms',
         'Current_state', 'Age', 'city', 'Price']]
df.head()

### Bilan de l’étape 6

Les noms et l’ordre des colonnes sont désormais harmonisés, ce qui facilite les contrôles de qualité et la préparation du modèle.

### Figure 5 — Distribution initiale des variables principales

Les distributions permettent de repérer l’asymétrie des surfaces et des prix, ainsi que la fréquence des différentes catégories. Elles sont observées avant les filtres afin de comprendre pourquoi un contrôle des valeurs extrêmes est nécessaire.

In [ ]:
# Distributions numériques et catégorielles
for col in ['Area', 'Price']:
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    s = s[s > 0]
    fig = px.histogram(x=np.log10(s), nbins=40, title=f'{col} (échelle log10)',
                       labels={'x':f'log10({col})','y':"Nombre d'annonces"})
    fig.update_traces(marker_color=BLUE)
    fig.update_layout(template='plotly_white')
    fig.show()

for col in ['Rooms', 'Bedrooms', 'Bathrooms']:
    s = df[col].dropna()
    fig = px.histogram(x=s, title=f'Distribution de {col}', labels={'x':col,'y':"Nombre d'annonces"})
    fig.update_traces(marker_color=BLUE)
    fig.update_layout(bargap=.15, template='plotly_white')
    fig.show()

vc = df['Type'].fillna('Manquant').value_counts().rename_axis('Type').reset_index(name='Nombre')
fig = px.bar(vc, x='Type', y='Nombre', text='Nombre', title='Répartition par type de bien')
fig.update_traces(marker_color=BLUE, textposition='outside')
fig.update_layout(xaxis_tickangle=-30, template='plotly_white')
fig.show()

### Interprétation de la figure 5

Les histogrammes des variables numériques permettent d’observer l’étendue et l’asymétrie des prix et des surfaces ; les graphiques catégoriels montrent la composition de l’échantillon. Les distributions fortement étirées justifient une inspection des valeurs extrêmes. À ce stade, les observations n’ont pas encore subi les filtres décrits ensuite.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

## 7. Filtrage des valeurs extrêmes

Des bornes plausibles sont appliquées au prix, à la surface et au nombre de pièces/salles. Ces seuils visent à éliminer les valeurs manifestement aberrantes ou incompatibles avec le périmètre de l’étude. Ils constituent des choix méthodologiques qui doivent être documentés dans le rapport.

In [ ]:
before7 = df.copy()                      # snapshot pour la visualisation

df = df[(df['Price'] >= 100_000) & (df['Price'] <= 60_000_000)]
df = df[(df['Area'] >= 15) & (df['Area'] <= 3000)]
for c in ['Rooms', 'Bedrooms', 'Bathrooms']:
    df = df[(df[c].isna()) | ((df[c] >= 0) & (df[c] <= 20))]

funnel['Après filtre outliers'] = len(df)
print("Après filtre outliers:", df.shape)

### Bilan de l’étape 7

Le filtrage écarte les annonces situées hors des plages définies. Les seuils doivent être mentionnés explicitement dans le rapport afin de rendre la démarche reproductible.

### Figure 6 — Effet du premier filtrage

Le graphique détaille les motifs de suppression et permet d’évaluer l’impact de chaque règle sur le volume d’annonces conservées.

In [ ]:
b = before7
kept7 = b.index.isin(df.index)

rules = pd.Series({
    'Price manquant':        b['Price'].isna().sum(),
    'Price < 100 000':       (b['Price'] < 100_000).sum(),
    'Price > 60 M':          (b['Price'] > 60_000_000).sum(),
    'Area manquante':        b['Area'].isna().sum(),
    'Area < 15 m²':          (b['Area'] < 15).sum(),
    'Area > 3000 m²':        (b['Area'] > 3000).sum(),
    'Pièces > 20 ou < 0':    sum(((b[c] > 20) | (b[c] < 0)).sum() for c in ['Rooms', 'Bedrooms', 'Bathrooms']),
}).sort_values()

ok = b[kept7].dropna(subset=['Area', 'Price']).copy()
ok['Statut'] = f"Conservées ({kept7.sum()})"
ko = b[~kept7].dropna(subset=['Area', 'Price']).copy()
ko = ko[(ko['Area'] > 0) & (ko['Price'] > 0)]
ko['Statut'] = f"Supprimées ({(~kept7).sum()})"
plot_df = pd.concat([ok, ko], axis=0)
plot_df = plot_df[(plot_df['Area'] > 0) & (plot_df['Price'] > 0)]
fig = px.scatter(plot_df, x='Area', y='Price', color='Statut', log_x=True, log_y=True,
                 opacity=.65, title='Price vs Area (log-log) — lignes conservées vs supprimées',
                 labels={'Area':'Area (m²)','Price':'Price (MAD)'})
fig.add_hline(y=100_000, line_dash='dash', line_color='black')
fig.add_hline(y=60_000_000, line_dash='dash', line_color='black')
fig.add_vline(x=15, line_dash='dot', line_color='black')
fig.add_vline(x=3000, line_dash='dot', line_color='black')
fig.update_layout(template='plotly_white')
fig.show()

rules_df = rules.rename_axis('Règle').reset_index(name='Nombre de lignes')
fig = px.bar(rules_df, x='Nombre de lignes', y='Règle', orientation='h', text='Nombre de lignes',
             title='Lignes concernées par chaque règle (une ligne peut violer plusieurs règles)')
fig.update_traces(marker_color=RED, textposition='outside')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_white')
fig.show()

### Interprétation de la figure 6

Les motifs de suppression permettent de comprendre quelles règles contribuent le plus à la réduction du jeu de données. Les observations écartées ne doivent pas être considérées automatiquement comme des erreurs certaines : les seuils sont des critères de nettoyage choisis pour le périmètre de l’étude et doivent être explicités dans le rapport.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

## 8. Traitement des valeurs manquantes et des doublons

Les lignes incomplètes sur les variables essentielles à l’apprentissage sont supprimées. Les variables considérées comme optionnelles sont complétées selon les règles du notebook, puis les doublons sont retirés afin de limiter les répétitions dans le jeu d’apprentissage.

In [ ]:
essential = ['Type', 'Area', 'Rooms', 'Bedrooms', 'Bathrooms', 'city', 'Price']
optional  = ['Current_state', 'Age', 'Localisation']

miss_before8 = df[essential + optional].isna().sum().sort_values()
n_before8 = len(df)

df_clean = df.dropna(subset=essential).copy()
funnel['Après dropna (obligatoires)'] = len(df_clean)
print("Après dropna colonnes essentielles:", df_clean.shape)

df_clean['Current_state'] = df_clean['Current_state'].fillna('Inconnu')
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Localisation'] = df_clean['Localisation'].fillna(df_clean['city'])

df_clean.drop_duplicates(inplace=True)
funnel['Après doublons'] = len(df_clean)
print("Après suppression doublons:", df_clean.shape)

### Bilan de l’étape 8

Les variables essentielles sont complètes pour les annonces conservées ; les champs optionnels sont traités selon les règles prévues et les doublons sont supprimés.

### Figure 7 — Valeurs manquantes : variables essentielles et optionnelles

La comparaison permet de distinguer les variables dont l’absence entraîne l’exclusion d’une annonce de celles pour lesquelles une valeur de remplacement est utilisée.

In [ ]:
cols_c = [RED if c in essential else BLUE for c in miss_before8.index]
miss_df = miss_before8.rename_axis('Colonne').reset_index(name='Valeurs manquantes')
miss_df['Traitement'] = miss_df['Colonne'].apply(lambda c: 'Obligatoire — ligne supprimée' if c in essential else 'Optionnelle — valeur remplie')
fig = px.bar(miss_df, x='Valeurs manquantes', y='Colonne', orientation='h', color='Traitement', text='Valeurs manquantes',
             title='Valeurs manquantes avant dropna')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_range=[0, max(1, miss_before8.max()*1.15)], yaxis={'categoryorder':'total ascending'}, template='plotly_white')
fig.show()

steps8 = pd.Series({'Avant dropna': n_before8,
                    'Après dropna': funnel['Après dropna (obligatoires)'],
                    'Après doublons': funnel['Après doublons']})
steps_df = steps8.rename_axis('Étape').reset_index(name='Nombre de lignes')
fig = px.bar(steps_df, x='Étape', y='Nombre de lignes', text='Nombre de lignes', title='Évolution du nombre de lignes')
fig.update_traces(marker_color=[GREY, BLUE, GREEN], textposition='outside')
fig.update_layout(yaxis_range=[0, steps8.max()*1.12], template='plotly_white')
fig.show()

### Interprétation de la figure 7

Les valeurs manquantes des variables essentielles sont particulièrement importantes, car elles empêchent l’utilisation directe d’une annonce pour l’apprentissage. Pour les variables optionnelles, l’imputation conserve davantage d’observations, au prix d’une information parfois moins précise. Le traitement doit être évalué en tenant compte de ce compromis.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

## 9. Contrôle des prix au mètre carré par type de bien

Le prix au mètre carré (`Price / Area`) est examiné séparément pour chaque type de bien. Une règle fondée sur l’intervalle interquartile (IQR) permet de repérer les observations atypiques sans appliquer exactement la même distribution à toutes les catégories.

In [ ]:
df_clean['ppm2'] = df_clean['Price'] / df_clean['Area']

bounds = df_clean.groupby('Type')['ppm2'].quantile([0.25, 0.75]).unstack()
bounds['iqr']  = bounds[0.75] - bounds[0.25]
bounds['low']  = bounds[0.25] - 0.5 * bounds['iqr']
bounds['high'] = bounds[0.75] + 0.5 * bounds['iqr']

low_map  = df_clean['Type'].map(bounds['low'])
high_map = df_clean['Type'].map(bounds['high'])
mask9 = (df_clean['ppm2'] >= low_map) & (df_clean['ppm2'] <= high_map)

before9 = df_clean.copy()
before9['statut'] = np.where(mask9, 'conservée', 'supprimée')

df_clean = df_clean[mask9].drop(columns=['ppm2'])
funnel['Après filtre prix/m²'] = len(df_clean)
print("Après filtre prix/m² (IQR par Type):", df_clean.shape)

### Bilan de l’étape 9

Le contrôle du prix au mètre carré complète les bornes globales en tenant compte des différences entre types de biens.

### Figure 8 — Détection des prix au mètre carré atypiques

Le nuage de points compare les annonces conservées et celles écartées. La lecture doit tenir compte du type de bien, car les niveaux de prix au mètre carré peuvent différer entre catégories.

In [ ]:
pal = {'conservée': GREEN, 'supprimée': RED}
types = list(bounds.index)

fig = px.strip(before9, x='Type', y='ppm2', color='statut', category_orders={'Type':types},
               color_discrete_map=pal, hover_data=['Area','Price'],
               title='Prix au m² par type — lignes pointillées = bornes IQR')
fig.update_yaxes(type='log', title='MAD / m²')
for t in types:
    fig.add_hline(y=max(bounds.loc[t, 'low'], 1), line_dash='dash', line_color='black', opacity=.35)
fig.update_layout(template='plotly_white')
fig.show()

fig = px.scatter(before9, x='Area', y='Price', color='statut', color_discrete_map=pal,
                 opacity=.7, log_x=True, log_y=True,
                 title=f"Price vs Area : {(before9['statut'] == 'supprimée').sum()} lignes supprimées",
                 labels={'Area':'Area (m²)','Price':'Price (MAD)'})
fig.update_layout(template='plotly_white')
fig.show()

### Interprétation de la figure 8

Les points signalés comme atypiques sont comparés aux annonces conservées à l’intérieur de chaque type de bien. Le filtre réduit l’influence potentielle de valeurs au mètre carré extrêmes, mais peut aussi retirer des biens réellement haut de gamme. Il est donc important de présenter la règle IQR et son coefficient dans le rapport.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

## 10. Export du jeu de données nettoyé

Le jeu nettoyé est exporté au format CSV pour être réutilisé indépendamment du notebook. Ce fichier constitue la sortie principale de la phase de préparation des données.

In [ ]:
df_clean.to_csv('mubawab_clean.csv', index=False)
df_clean.head()

### Bilan de l’étape 10

Le fichier `mubawab_clean.csv` constitue le jeu nettoyé exportable et réutilisable pour d’autres analyses.

### Figure 9 — Bilan global du nettoyage

L’entonnoir récapitule le nombre d’annonces restantes après les étapes successives. Il permet de mesurer l’ampleur du nettoyage et de vérifier que les règles n’ont pas réduit le jeu de données de manière inattendue.

In [ ]:
# (a) Entonnoir : nombre de lignes à chaque étape
fs = pd.Series(funnel)
fs_df = fs.rename_axis('Étape').reset_index(name='Nombre de lignes')
fs_df['Pourcentage conservé'] = fs_df['Nombre de lignes'] / fs.iloc[0] * 100
fig = px.bar(fs_df.iloc[::-1], x='Nombre de lignes', y='Étape', orientation='h',
             text=fs_df.iloc[::-1]['Nombre de lignes'], hover_data=['Pourcentage conservé'],
             title="Entonnoir de nettoyage : lignes conservées à chaque étape")
fig.update_traces(marker_color=BLUE, textposition='outside')
fig.update_layout(xaxis_range=[0, fs.max()*1.2], template='plotly_white')
fig.show()

# (b) Distributions brut vs nettoyé
def overlay_plotly(col, log=False, discrete=False):
    a = pd.to_numeric(df_raw[col.lower()], errors='coerce').dropna()
    c = df_clean[col].dropna()
    if log:
        a, c = a[a > 0], c[c > 0]
        a, c = np.log10(a), np.log10(c)
    if discrete:
        a = a[a <= 20]
    plot_df = pd.concat([pd.DataFrame({'Valeur':a, 'Jeu':'Brut'}),
                         pd.DataFrame({'Valeur':c, 'Jeu':'Nettoyé'})], ignore_index=True)
    fig = px.histogram(plot_df, x='Valeur', color='Jeu', histnorm='probability density',
                       barmode='overlay', opacity=.55, nbins=40,
                       title=col + (' (log10)' if log else ''),
                       color_discrete_map={'Brut':RED,'Nettoyé':GREEN})
    fig.update_layout(template='plotly_white')
    fig.show()

for col, log, discrete in [('Area',True,False),('Price',True,False),
                           ('Rooms',False,True),('Bedrooms',False,True),('Bathrooms',False,True)]:
    overlay_plotly(col, log=log, discrete=discrete)

t_raw = df_raw['type'].fillna('Manquant').value_counts(normalize=True) * 100
t_cln = df_clean['Type'].value_counts(normalize=True) * 100
tt = pd.DataFrame({'Type':sorted(set(t_raw.index).union(t_cln.index))})
tt['Brut (%)'] = tt['Type'].map(t_raw).fillna(0)
tt['Nettoyé (%)'] = tt['Type'].map(t_cln).fillna(0)
tt_long = tt.melt(id_vars='Type', var_name='Jeu', value_name='Pourcentage')
fig = px.bar(tt_long, x='Type', y='Pourcentage', color='Jeu', barmode='group',
             color_discrete_map={'Brut (%)':RED,'Nettoyé (%)':GREEN},
             title='Répartition des types : données brutes vs nettoyées')
fig.update_layout(xaxis_tickangle=-30, template='plotly_white')
fig.show()

### Interprétation de la figure 9

L’entonnoir permet de suivre l’évolution de la taille du jeu de données tout au long du pipeline. Une baisse progressive est attendue lors des filtres et du traitement des valeurs manquantes. Le volume final doit rester suffisant pour entraîner et évaluer un modèle ; les effectifs affichés après exécution doivent être repris dans le rapport.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

<div style="background:#5B4B8A;color:white;padding:16px 20px;border-radius:10px;margin:10px 0;">
<h1 style="margin:0;color:white;">PARTIE III — PRÉTRAITEMENT POUR LE MACHINE LEARNING</h1>
<p style="margin:6px 0 0;">Objectif : définir la cible, sélectionner les variables explicatives, regrouper les catégories rares et préparer l’encodage dans un pipeline.</p>
</div>

### Conclusion de la partie II — Nettoyage et feature engineering

Les données ont été structurées et harmonisées, les valeurs manquantes traitées selon des règles définies, les doublons supprimés et les observations atypiques filtrées à partir de seuils explicites. La variable `Age` a également été dérivée de l’état déclaré du bien. **À retenir :** le jeu nettoyé exporté dans `mubawab_clean.csv` sert de base aux étapes de préparation et de modélisation.

## 11. Préparation des variables pour le modèle

Les variables explicatives (`X`) et la cible (`y = Price`) sont séparées. Comme les annonces concernent Rabat, la variable `city` est constante et n’apporte pas d’information discriminante au modèle. Les localisations trop rares sont regroupées afin de limiter la création de catégories très peu représentées.

In [ ]:
X = df_clean[['Type', 'Localisation', 'Area', 'Rooms', 'Bedrooms', 'Bathrooms',
              'Current_state', 'Age']].copy()
y = df_clean['Price']

counts = X['Localisation'].value_counts()
rare = counts[counts < 3].index
X['Localisation'] = X['Localisation'].where(~X['Localisation'].isin(rare), 'Autre')

cat_cols = ['Type', 'Localisation', 'Current_state']
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
], remainder='passthrough')

print(f"Localisation : {counts.shape[0]} quartiers -> {X['Localisation'].nunique()} après regroupement")

### Bilan de l’étape 11

Les variables explicatives et la cible sont séparées. Le regroupement des localisations rares réduit la fragmentation des catégories avant l’encodage.

### Figure 10 — Fréquence des quartiers et regroupement des catégories rares

La figure montre quelles localisations sont suffisamment représentées et lesquelles sont regroupées sous la catégorie `Autre`. Ce regroupement réduit la granularité des catégories rares et rend l’encodage plus robuste.

In [ ]:
cs = counts.sort_values().rename_axis('Quartier').reset_index(name='Nombre d’annonces')
cs['Statut'] = cs['Nombre d’annonces'].apply(lambda v: 'Moins de 3 annonces' if v < 3 else '3 annonces ou plus')
fig = px.bar(cs, x='Nombre d’annonces', y='Quartier', orientation='h', color='Statut',
             color_discrete_map={'Moins de 3 annonces':RED,'3 annonces ou plus':BLUE},
             title="Annonces par quartier — les quartiers rares sont regroupés dans 'Autre'")
fig.add_vline(x=3, line_dash='dash', line_color='black')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_white')
fig.show()

info = pd.Series({'Quartiers avant': counts.shape[0],
                  'Quartiers après': X['Localisation'].nunique(),
                  'Lignes dans "Autre"': (X['Localisation'] == 'Autre').sum()})
info_df = info.rename_axis('Indicateur').reset_index(name='Valeur')
fig = px.bar(info_df, x='Indicateur', y='Valeur', text='Valeur', title='Effet du regroupement des quartiers')
fig.update_traces(textposition='outside', marker_color=[GREY,GREEN,RED])
fig.update_layout(xaxis_tickangle=-20, template='plotly_white')
fig.show()

### Interprétation de la figure 10

Les localisations ayant moins de trois occurrences sont regroupées dans `Autre`, tandis que les catégories plus fréquentes sont conservées. Cette règle limite les catégories peu documentées dans l’encodage, mais réduit le niveau de détail géographique pour les quartiers rares.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

<div style="background:#9A5B19;color:white;padding:16px 20px;border-radius:10px;margin:10px 0;">
<h1 style="margin:0;color:white;">PARTIE IV — MODÉLISATION ET ÉVALUATION</h1>
<p style="margin:6px 0 0;">Objectif : entraîner le Random Forest, mesurer ses performances et vérifier leur stabilité à l’aide de la validation croisée.</p>
</div>

### Conclusion de la partie III — Prétraitement

Les variables explicatives `X` et la cible `y = Price` sont séparées. Les localisations peu fréquentes sont regroupées dans `Autre`, puis les variables catégorielles sont encodées avec `OneHotEncoder`. Le `ColumnTransformer` et le `Pipeline` assurent que les mêmes transformations sont appliquées pendant l’entraînement et la prédiction.

## 12. Entraînement du modèle Random Forest

Le prix présente une distribution fortement asymétrique. Le modèle est donc entraîné sur `log1p(Price)`, puis les prédictions sont reconverties en dirhams avec `expm1`. Le prétraitement et le modèle sont réunis dans un pipeline pour appliquer les mêmes transformations aux données d’entraînement et de test.

In [ ]:
model = Pipeline([
    ('prep', preprocess),
    ('rf', RandomForestRegressor(n_estimators=500, min_samples_leaf=2,
                                 random_state=42, n_jobs=-1))
])

y_log = np.log1p(y)

X_train, X_test, y_train, y_test, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
pred = np.expm1(model.predict(X_test))

r2 = r2_score(y_test_raw, pred)
mae = mean_absolute_error(y_test_raw, pred)
print(f"R2 (test set, split 80/20) = {r2:.4f}")
print(f"MAE = {mae:,.0f} MAD")

### Bilan de l’étape 12

Le pipeline regroupe prétraitement et Random Forest afin de limiter les écarts entre la préparation des données d’entraînement et celle des données de prédiction.

### Figure 11 — Comparaison entre prix réels et prix prédits

Les graphiques évaluent les prédictions sur le jeu de test sous plusieurs angles : proximité entre valeurs réelles et prédites, distribution des erreurs et comportement selon le type de bien.

In [ ]:
yt = y_test_raw.values
rel = (pred - yt) / yt * 100
typ = X_test['Type'].values

eval_df = pd.DataFrame({'Prix réel':yt, 'Prix prédit':pred, 'Type':typ, 'Erreur relative (%)':rel})
lim = max(eval_df['Prix réel'].max(), eval_df['Prix prédit'].max()) * 1.05
fig = px.scatter(eval_df, x='Prix réel', y='Prix prédit', color='Type', opacity=.7,
                 title=f'Prédit vs Réel (test) — R² = {r2:.3f}',
                 hover_data=['Erreur relative (%)'])
fig.add_trace(go.Scatter(x=[0,lim], y=[0,lim], mode='lines', name='Prédiction parfaite',
                         line=dict(color='red', dash='dash')))
fig.update_xaxes(range=[0,lim], tickformat='~s')
fig.update_yaxes(range=[0,lim], tickformat='~s')
fig.update_layout(template='plotly_white')
fig.show()

fig = px.histogram(eval_df, x='Erreur relative (%)', nbins=35, marginal='rug',
                   title=f"Distribution des erreurs — médiane |err| = {np.median(np.abs(rel)):.1f}%",
                   range_x=[-100,100])
fig.add_vline(x=0, line_dash='dash', line_color='red')
fig.update_layout(template='plotly_white')
fig.show()

fig = px.scatter(eval_df, x='Prix réel', y='Erreur relative (%)', color='Type', opacity=.65,
                 title='Erreur relative selon le prix réel')
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.update_yaxes(range=[-100,100])
fig.update_xaxes(tickformat='~s')
fig.update_layout(template='plotly_white')
fig.show()

### Interprétation de la figure 11

Sur le graphique réel/prédit, des points proches de la diagonale indiquent des estimations plus proches des prix observés. Les graphiques d’erreur permettent de repérer un biais systématique, des erreurs extrêmes ou des différences selon le type de bien. Il faut interpréter ces éléments conjointement avec les métriques calculées sur le jeu de test, et non à partir du seul aspect visuel.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

## 13. Validation croisée à cinq plis

La validation croisée répète l’évaluation sur cinq partitions différentes. Elle permet d’apprécier la stabilité des performances et de vérifier si le score obtenu sur un seul découpage est représentatif. Le seuil de R² de 0,8 est un objectif d’évaluation, pas une garantie de résultat.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)
cv_scores = []
y_arr = y.values
oof = np.zeros(len(y_arr))               # prédictions out-of-fold
fold_id = np.zeros(len(y_arr), dtype=int)

for k, (tr_idx, te_idx) in enumerate(kf.split(X), start=1):
    model.fit(X.iloc[tr_idx], np.log1p(y_arr[tr_idx]))
    p = np.expm1(model.predict(X.iloc[te_idx]))
    oof[te_idx] = p
    fold_id[te_idx] = k
    cv_scores.append(r2_score(y_arr[te_idx], p))

print("R2 par fold:", [round(s, 3) for s in cv_scores])
print(f"R2 moyen (5-fold CV) = {np.mean(cv_scores):.4f}")

### Bilan de l’étape 13

La validation croisée complète l’évaluation sur un seul jeu de test et donne une indication de la robustesse des résultats.

### Figure 12 — Stabilité des performances en validation croisée

La comparaison des scores par pli met en évidence leur dispersion. Un score moyen élevé accompagné de résultats relativement proches suggère une meilleure stabilité ; une forte variation invite à examiner la taille et la composition des partitions.

In [ ]:
# R² par fold
cv_df = pd.DataFrame({'Fold':[f'Fold {i+1}' for i in range(len(cv_scores))], 'R²':cv_scores})
fig = px.bar(cv_df, x='Fold', y='R²', text=cv_df['R²'].map(lambda v:f'{v:.3f}'),
             title='R² sur chaque fold de validation croisée')
fig.add_hline(y=.8, line_dash='dash', line_color='red', annotation_text='Objectif R² = 0.8')
fig.add_hline(y=float(np.mean(cv_scores)), line_dash='dot', line_color='navy',
              annotation_text=f'Moyenne = {np.mean(cv_scores):.3f}')
fig.update_traces(marker_color=GREEN, textposition='outside')
fig.update_yaxes(range=[0,1.05])
fig.update_layout(template='plotly_white')
fig.show()

oof_df = pd.DataFrame({'Prix réel':y_arr, 'Prix prédit (out-of-fold)':oof, 'Fold':fold_id.astype(str)})
lim = max(np.max(y_arr), np.max(oof)) * 1.05
fig = px.scatter(oof_df, x='Prix réel', y='Prix prédit (out-of-fold)', color='Fold', opacity=.7,
                 title=f"Prédit vs Réel — toutes les annonces (R² global = {r2_score(y_arr, oof):.3f})")
fig.add_trace(go.Scatter(x=[0,lim], y=[0,lim], mode='lines', name='Prédiction parfaite',
                         line=dict(color='red', dash='dash')))
fig.update_xaxes(range=[0,lim], tickformat='~s')
fig.update_yaxes(range=[0,lim], tickformat='~s')
fig.update_layout(template='plotly_white')
fig.show()

### Interprétation de la figure 12

Les scores des cinq plis montrent dans quelle mesure les performances dépendent de la partition utilisée. Des scores proches indiquent une stabilité relative ; un pli nettement plus faible peut révéler une sensibilité à certaines annonces ou catégories. Le R² moyen et sa dispersion doivent être rapportés avec les valeurs réellement obtenues.

> **À retenir :** cette figure sert à contrôler la qualité de l’étape et à justifier la décision de préparation associée.

<div style="background:#34495E;color:white;padding:16px 20px;border-radius:10px;margin:10px 0;">
<h1 style="margin:0;color:white;">PARTIE V — MODÈLE FINAL ET UTILISATION</h1>
<p style="margin:6px 0 0;">Objectif : réentraîner le pipeline final, sauvegarder le modèle et illustrer son utilisation sur une annonce exemple.</p>
</div>

### Conclusion de la partie IV — Modélisation et évaluation

Le modèle `RandomForestRegressor` est entraîné sur la cible transformée par `log1p`, puis les prédictions sont reconverties en MAD pour l’évaluation. Le R² mesure la qualité d’ajustement prédictif et la MAE exprime l’erreur moyenne en dirhams. La validation croisée à cinq plis complète l’évaluation du jeu de test en examinant la variabilité des scores.

## 14. Réentraînement final et sauvegarde du modèle

Après l’évaluation, le pipeline est réentraîné sur l’ensemble des données préparées puis sauvegardé au format Joblib. Le fichier généré pourra être rechargé dans une autre application Python, sous réserve de conserver un environnement compatible.

In [ ]:
best_catboost = model["CatBoost"]
joblib.dump(best_catboost, 'modele_MaisonDeLUX.pkl')

### Bilan de l’étape 14

Le pipeline final est entraîné sur toutes les données disponibles, puis enregistré dans `rf_price_model.joblib` pour une réutilisation ultérieure.

## 15. Exemple d’estimation du prix

Une annonce fictive de démonstration est construite avec les variables attendues par le pipeline. Le modèle produit une estimation en dirhams. Cette valeur illustre l’utilisation technique du modèle et ne constitue pas une expertise immobilière ni une garantie du prix de marché.

In [ ]:
example = pd.DataFrame([{
    'Type': 'appartement', 'Localisation': 'Agdal', 'Area': 110,
    'Rooms': 3, 'Bedrooms': 2, 'Bathrooms': 1,
    'Current_state': '5-10 ans', 'Age': 7.5,
}])

pred_price = np.expm1(model.predict(example))[0]
print(f"Prix estimé : {pred_price:,.0f} MAD")

### Bilan de l’étape 15

L’exemple confirme la forme attendue des variables d’entrée et illustre le retour d’une estimation en MAD. Pour une utilisation réelle, il faut fournir des caractéristiques cohérentes avec celles observées dans les données d’entraînement.

### Conclusion de la partie V — Utilisation du modèle

Le pipeline final est réentraîné sur les données préparées puis sauvegardé dans `rf_price_model.joblib`. L’exemple de prédiction illustre le format des caractéristiques nécessaires pour estimer un prix. Pour une utilisation réelle, il faut fournir des valeurs cohérentes avec les catégories et les plages observées dans le jeu d’entraînement.

# Conclusion générale

Ce notebook met en place une chaîne de traitement complète pour l’estimation des prix immobiliers à Rabat : diagnostic des données, sélection des variables, structuration de la localisation, conversion de l’âge du bien, filtrage des valeurs atypiques, traitement des données incomplètes, préparation des variables et entraînement d’un modèle Random Forest.

Le nettoyage vise à améliorer la cohérence des annonces utilisées pour l’apprentissage. En contrepartie, les seuils appliqués et le regroupement des quartiers modifient le périmètre et le niveau de détail des données ; ces choix doivent donc être transparents dans le rapport.

L’évaluation repose sur les prédictions du jeu de test et sur une validation croisée à cinq plis. Les conclusions sur la qualité du modèle doivent être formulées à partir des métriques réellement obtenues après exécution, notamment le R², les erreurs de prédiction et la stabilité entre les plis. L’objectif R² > 0,8 ne doit être présenté comme atteint que si les résultats le confirment.

Enfin, le modèle sauvegardé peut servir de base à un outil d’estimation exploratoire. Ses prédictions restent dépendantes de la qualité, de la représentativité et de l’actualité des annonces disponibles ; elles ne remplacent pas une expertise immobilière ni une validation sur des transactions réelles.
### Méthodologie de preprocessing et d'optimisation du modèle

La préparation des données a commencé par la sélection des variables pertinentes pour l'estimation des prix immobiliers à Rabat. Les informations de localisation ont été structurées afin de distinguer le quartier de la ville, tandis que la variable catégorielle relative à l'état du bien a été convertie en une estimation numérique de son âge.

Le nettoyage a ensuite reposé sur plusieurs opérations : filtrage des valeurs extrêmes selon des seuils prédéfinis, détection des annonces atypiques à partir du prix au mètre carré et de la méthode IQR par type de bien, traitement des valeurs manquantes selon la nature des variables et suppression des doublons. Les quartiers peu représentés ont également été regroupés dans une catégorie « Autre ».

Pour préparer les données à la modélisation, les variables catégorielles ont été encodées avec One-Hot Encoding. Les transformations ont été organisées à l'aide de `ColumnTransformer` et `Pipeline`, puis le prix cible a été transformé par la fonction logarithmique `log1p` afin de réduire son asymétrie.

Le modèle choisi est un `RandomForestRegressor` composé de 500 arbres, avec une séparation initiale de 80 % des données pour l'entraînement et de 20 % pour le test. Ses performances sont évaluées à l'aide du coefficient de détermination R² et de l'erreur absolue moyenne (MAE). Une validation croisée à cinq plis complète cette évaluation afin d'examiner la stabilité des résultats.

Ces choix méthodologiques visent à améliorer la cohérence des données et la capacité de généralisation du modèle. Leur contribution réelle à l'amélioration des performances doit toutefois être confirmée par une comparaison expérimentale et une évaluation sans fuite de données.
